# Lab 74: The Thatcher Illusion

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-74.ipynb)

**What you will do:** rate how grotesque a Thatcherized face looks upright versus upside down, compare the two ratings, and test whether an AI face-landmark model shows the same orientation-dependent blind spot on images it generates itself.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 74 you cut out the eyes and mouth of a face photo, pasted them back in upside down, and
compared the result upright versus inverted. Inverted, the edit looks "more or less normal."
Upright, the same edit looks grotesque. This is the **Thatcher illusion** (Thompson, 1980): face
perception is holistic and orientation-specific, so the brain only notices the impossible
configuration when the face is the right way up.

## Record your results

Rate how grotesque the Thatcherized face looks (0 = perfectly normal, 10 = monstrous) in both
orientations, from yourself and a few friends who have not seen the original photo.

In [ ]:
# example data: replace with yours
ratings = pd.DataFrame({
    "person": ["You", "Friend 1", "Friend 2", "Friend 3"],
    "thatcherized_upright_grotesqueness": [9, 8, 10, 7],
    "thatcherized_inverted_grotesqueness": [2, 1, 3, 2],
})
display(ratings)
ratings.mean(numeric_only=True).plot.bar(color=["firebrick", "steelblue"], figsize=(5, 3.5))
plt.ylabel("mean grotesqueness (0-10)"); plt.ylim(0, 10); plt.xticks(rotation=15); plt.show()

## Compare

Lab 74 describes the inverted version as looking "more or less normal" and the upright
version as "grotesque" -- the same edit, noticed only right-side up. Check your ratings
against that pattern.

In [ ]:
upright = ratings.thatcherized_upright_grotesqueness.mean()
inverted = ratings.thatcherized_inverted_grotesqueness.mean()
print(f"Upright: {upright:.1f}/10 grotesque. Inverted: {inverted:.1f}/10 grotesque.")
if upright > inverted:
    print("This matches the predicted inversion effect: the same manipulation is far more")
    print("noticeable right-side up than upside down.")
else:
    print("This does not show the usual pattern -- check that the eyes and mouth are actually")
    print("rotated 180 degrees within the face, not the whole photo.")

## The AI side

The book asks whether an AI face system shares this orientation-dependent blind spot. Below,
a face-landmark model (MediaPipe Face Mesh) is tested on four images the notebook generates
itself from one face: original and Thatcherized, each upright and inverted -- avoiding any
need to use a real, identifiable person's photo for the manipulation.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q mediapipe

import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from sklearn.datasets import fetch_olivetti_faces

# The Olivetti faces: AT&T Laboratories Cambridge's "Database of Faces" (Samaria & Harter,
# 1994), released for research and distributed with scikit-learn. Please acknowledge the
# source if you reuse the images.
faces = fetch_olivetti_faces()
face = (faces.images[0] * 255).astype(np.uint8)
face = cv2.resize(face, (256, 256), interpolation=cv2.INTER_CUBIC)
face_rgb = cv2.cvtColor(face, cv2.COLOR_GRAY2RGB)

def thatcherize(img):
    # Cut out the middle band (eyes and mouth) and rotate it 180 degrees in place --
    # the same operation the book describes doing with scissors and glue.
    out = img.copy()
    band = out[85:180, 40:216]
    out[85:180, 40:216] = np.rot90(band, 2)
    return out

variants = {
    "original, upright": face_rgb,
    "thatcherized, upright": thatcherize(face_rgb),
    "original, inverted": np.rot90(face_rgb, 2),
    "thatcherized, inverted": np.rot90(thatcherize(face_rgb), 2),
}

!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
base_options = mp_python.BaseOptions(model_asset_path="face_landmarker.task")
landmarker = mp_vision.FaceLandmarker.create_from_options(mp_vision.FaceLandmarkerOptions(base_options=base_options))

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (name, img) in zip(axes, variants.items()):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(img))
    result = landmarker.detect(mp_image)
    ax.imshow(img)
    if result.face_landmarks:
        h, w = img.shape[:2]
        xs = [lm.x * w for lm in result.face_landmarks[0]]
        ys = [lm.y * h for lm in result.face_landmarks[0]]
        ax.scatter(xs, ys, c="lime", s=1)
        ax.set_title(f"{name}\n(face found)")
    else:
        ax.set_title(f"{name}\n(no face found)")
    ax.axis("off")
plt.tight_layout(); plt.show()

However grotesque the upright thatcherized face looks to you, Face Mesh finds landmarks in
roughly the same places across all four images: its geometric template-matching has no
orientation-specific "holistic" stage to disrupt, so it cannot be shocked by a configuration
that violates a face template it never held in the human sense. If anything, this kind of
model is more likely to struggle with the inverted images generally (they look less like the
upright, front-facing training photos it saw) than with the Thatcherized manipulation
specifically -- the opposite pattern from the human effect, where inversion hides the
manipulation rather than revealing it.

## Pool the class data

Pool the class's grotesqueness ratings for both orientations to see the inversion effect on
a larger, less noisy sample.

In [ ]:
import io
csv_text = """id,upright_grotesqueness,inverted_grotesqueness
P01,9,2
P02,8,3
P03,10,1
P04,7,4
P05,9,2
P06,6,3
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
diff = class_df.upright_grotesqueness - class_df.inverted_grotesqueness
boot = [diff.sample(len(diff), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean grotesqueness -- upright: {class_df.upright_grotesqueness.mean():.1f}/10,",
      f"inverted: {class_df.inverted_grotesqueness.mean():.1f}/10")
print(f"Mean inversion effect: {diff.mean():.1f} points (95% bootstrap interval {lo:.1f} to {hi:.1f})")
class_df[["upright_grotesqueness", "inverted_grotesqueness"]].mean().plot.bar(
    color=["firebrick", "steelblue"], figsize=(5, 3.5))
plt.ylabel("mean grotesqueness (0-10)"); plt.ylim(0, 10); plt.xticks(rotation=15); plt.show()

## Questions to think about

1. The FFA responds more strongly to upright faces than inverted ones, and its activity tracks the ability to detect configurational changes. Based on your own ratings above, does your subjective sense of "how grotesque" track your confidence in spotting exactly what was changed, or can the two come apart?
2. The book notes that orientation-specific expertise also applies to other categories, such as dog breeds for dog experts. Is there a category you personally have deep expertise in (a sport, a hobby, a set of faces from your own life) where you suspect you would show an unusually large inversion effect?
3. If Face Mesh in the demo above found landmarks equally well on all four images, what does that tell you about the difference between "detecting where the facial features are" and "judging whether their configuration looks normal"?
4. The book raises inverted viewing as a possible deepfake-detection trick, then notes that real detection tools instead look for digital artefacts at blend boundaries. Given what you saw above, why might training a detector to mimic your inversion-sensitive holistic processing be harder than training it to spot pixel-level fingerprints?

## Challenge

Try the illusion on a face you know very well (a close friend or family member, with their
permission) rather than a celebrity or stranger. Does deep familiarity with a specific face make
the upright Thatcherized version look even more grotesque, or does it make no difference to how
strongly the illusion hits you?

In [ ]:
# Example data: replace with your own ratings for a familiar face versus an unfamiliar one,
# both Thatcherized and viewed upright.
familiarity_ratings = pd.DataFrame({
    "face": ["Familiar (friend/family)", "Unfamiliar (stranger/celebrity)"],
    "upright_grotesqueness": [10, 8],   # example data
})
display(familiarity_ratings)
print("Compare these two ratings with your original upright rating earlier in this notebook,")
print("and with the inverted ratings, to see whether familiarity changes the size of the effect.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-74.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")